# ⚡ Aelionix Academy — 1-Click Live AI Tutor (Fast Inference Serving)

This notebook loads your fine-tuned **Aelionix Tutor** adapter directly from Hugging Face Hub (`sagelord001/aelionix-tutor-qlora`) and spins up a live, free **OpenAI-compatible inference server** with a **Cloudflare Tunnel** in under **90 seconds**!

No dataset upload or retraining needed—just click **Runtime → Run all** and paste the generated URL into your `.env.local`!

### ⚙️ Step 0: Ensure Free T4 GPU is Enabled
In the top menu, go to: **Runtime** -> **Change runtime type** -> select **T4 GPU** -> click **Save**.

## Step 1: Install Serving Dependencies
Installs lightweight PyTorch quantization tools (`bitsandbytes`, `peft`, `transformers`), FastAPI, Uvicorn, and Cloudflare Tunnel.

In [ ]:
!pip install -q transformers peft bitsandbytes accelerate fastapi uvicorn pydantic

print("📥 Downloading Cloudflare Tunnel binary...")
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1
print("✅ Dependencies & Cloudflare Tunnel installed successfully!")

## Step 2: Load Model & Hugging Face Adapter directly from Cloud
Loads `Qwen/Qwen2.5-Coder-7B-Instruct` in 4-bit NF4 with KV-caching enabled, then applies your published adapter `sagelord001/aelionix-tutor-qlora`.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

BASE_MODEL = "Qwen/Qwen2.5-Coder-7B-Instruct"
ADAPTER_REPO = "sagelord001/aelionix-tutor-qlora"

print(f"[1/2] Loading base model: {BASE_MODEL} with 4-bit quantization...")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(ADAPTER_REPO, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
    trust_remote_code=True,
)

print(f"[2/2] Loading fine-tuned adapter directly from Hugging Face: {ADAPTER_REPO}...")
model = PeftModel.from_pretrained(base_model, ADAPTER_REPO)
model.eval()
if hasattr(model, "config"):
    model.config.use_cache = True

print("🎉 Model and adapter loaded successfully and ready for inference!")

## Step 3: Launch Live OpenAI-Compatible API Server + Cloudflare Tunnel
Spins up the FastAPI `/v1/chat/completions` endpoint and creates a public HTTPS tunnel. Keep this cell running while testing!

In [ ]:
import os
import re
import sys
import time
import threading
from typing import Optional, Any
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, ConfigDict
import uvicorn

app = FastAPI(title="Aelionix Tutor Live Server")

class ChatMessage(BaseModel):
    role: str
    content: str

class ChatCompletionRequest(BaseModel):
    model_config = ConfigDict(extra="allow")
    model: str = "aelionix-tutor"
    messages: list[ChatMessage]
    temperature: float = 0.35
    max_tokens: int = 450
    response_format: Optional[Any] = None

@app.get("/health")
def health():
    return {"status": "ok", "adapter": ADAPTER_REPO}

@app.post("/v1/chat/completions")
@app.post("/chat/completions")
def chat_completions(req: ChatCompletionRequest):
    if model is None or tokenizer is None:
        raise HTTPException(status_code=503, detail="Model not initialized")

    formatted = [{"role": m.role, "content": m.content} for m in req.messages]
    prompt = tokenizer.apply_chat_template(
        formatted,
        tokenize=False,
        add_generation_prompt=True,
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=min(req.max_tokens, 450),
            temperature=req.temperature,
            do_sample=req.temperature > 0.05,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
            use_cache=True,
        )
    generated = output_ids[0][inputs.input_ids.shape[1]:]
    reply = tokenizer.decode(generated, skip_special_tokens=True).strip()

    return {
        "id": f"chatcmpl-aelionix-{int(time.time())}",
        "object": "chat.completion",
        "created": int(time.time()),
        "model": req.model,
        "choices": [
            {
                "index": 0,
                "message": {"role": "assistant", "content": reply},
                "finish_reason": "stop",
            }
        ],
        "usage": {
            "prompt_tokens": inputs.input_ids.shape[1],
            "completion_tokens": len(generated),
            "total_tokens": inputs.input_ids.shape[1] + len(generated),
        },
    }

# Start FastAPI in a background thread
server_thread = threading.Thread(
    target=lambda: uvicorn.run(app, host="0.0.0.0", port=8000, log_level="warning"),
    daemon=True,
)
server_thread.start()
time.sleep(2)
print("✅ FastAPI server running on http://127.0.0.1:8000")

# Launch Cloudflare Tunnel in background logging to file
!nohup cloudflared tunnel --url http://127.0.0.1:8000 --logfile /content/tunnel.log > /dev/null 2>&1 &
print("🌐 Starting Cloudflare Tunnel...")

tunnel_url = None
for attempt in range(30):
    time.sleep(1)
    if os.path.exists("/content/tunnel.log"):
        with open("/content/tunnel.log", "r") as f:
            content = f.read()
        match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", content)
        if match:
            tunnel_url = match.group(0)
            break

if tunnel_url:
    print("\n" + "=" * 75)
    print("🎉 AELIONIX TUTOR LIVE INFERENCE SERVER IS READY!")
    print("=" * 75)
    print(f"Public API Base URL:  {tunnel_url}/v1")
    print("\nCopy and paste these lines into your local .env.local file:")
    print("-" * 75)
    print("AI_PROVIDER=openai-compatible")
    print(f"AI_BASE_URL={tunnel_url}/v1")
    print("AI_MODEL=aelionix-tutor")
    print("AI_API_KEY=colab-tunnel")
    print("AI_SUPPORTS_JSON_MODE=true")
    print("=" * 75)
    print("\nKeep this cell running while testing on your local machine!")
    while True:
        time.sleep(60)
else:
    print("⚠️ Could not extract Cloudflare URL. Check /content/tunnel.log.")
